In [58]:
import pandas as pd 
import numpy as np 

In [59]:
df= pd.read_csv("../data/raw/github_top_repositories_V2.csv")

In [60]:
df.head(2)

,Domain,Repository Name,Full Name,Description,Primary Language,Stars Count,Forks Count,Open Issues Count,Has Wiki,Has Pages,Has Projects,Size (KB),Created At,Updated At,Pushed At,Default Branch,Owner Login,Owner Type,License,Topics
0,Machine Learning,tensorflow,tensorflow/tensorflow,An Open Source Machine Learning Framework for ...,C++,194622,75263,4302,False,False,True,1302841,2015-11-07T01:19:20Z,2026-04-10T10:08:00Z,2026-04-10T10:13:20Z,master,tensorflow,Organization,Apache License 2.0,"deep-learning, deep-neural-networks, distribut..."
1,Machine Learning,transformers,huggingface/transformers,🤗 Transformers: the model-definition framework...,Python,159148,32816,2365,True,False,True,463713,2018-10-29T13:56:00Z,2026-04-10T10:08:23Z,2026-04-10T10:08:12Z,main,huggingface,Organization,Apache License 2.0,"audio, deep-learning, deepseek, gemma, glm, ha..."


**EDA**

In [61]:
df.shape

(5000, 20)

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Domain             5000 non-null   object
 1   Repository Name    5000 non-null   object
 2   Full Name          5000 non-null   object
 3   Description        5000 non-null   object
 4   Primary Language   5000 non-null   object
 5   Stars Count        5000 non-null   int64 
 6   Forks Count        5000 non-null   int64 
 7   Open Issues Count  5000 non-null   int64 
 8   Has Wiki           5000 non-null   bool  
 9   Has Pages          5000 non-null   bool  
 10  Has Projects       5000 non-null   bool  
 11  Size (KB)          5000 non-null   int64 
 12  Created At         5000 non-null   object
 13  Updated At         5000 non-null   object
 14  Pushed At          5000 non-null   object
 15  Default Branch     5000 non-null   object
 16  Owner Login        5000 non-null   object


In [63]:
# Missing Values

df.isna().sum()

Domain               0
Repository Name      0
Full Name            0
Description          0
Primary Language     0
Stars Count          0
Forks Count          0
Open Issues Count    0
Has Wiki             0
Has Pages            0
Has Projects         0
Size (KB)            0
Created At           0
Updated At           0
Pushed At            0
Default Branch       0
Owner Login          0
Owner Type           0
License              0
Topics               0
dtype: int64

In [64]:
# Checking Duplicates

df.duplicated().sum()

np.int64(3)

In [65]:
# Most common languages

df['Primary Language'].value_counts().head(10)

Primary Language
Python              1086
0                    602
C++                  486
JavaScript           462
Go                   436
Java                 370
Rust                 323
TypeScript           278
Jupyter Notebook     224
C                    107
Name: count, dtype: int64

*Python is the most commonly used language*

In [66]:
# Most common domains

df['Domain'].value_counts()

Domain
Machine Learning               200
Cybersecurity                  200
Robotics                       200
Internet of Things             200
Computer Vision                200
Natural Language Processing    200
Artificial Intelligence        200
Cloud Computing                200
Game Development               200
Backend Development            200
Frontend Development           200
DevOps                         200
Blockchain                     200
Deep Learning                  200
iOS                            200
Android                        200
Web Development                200
Data Science                   200
Rust                           200
Go                             200
C++                            200
Java                           200
JavaScript                     200
Python                         200
Software Engineering           200
Name: count, dtype: int64

*Different domains are balanced in the dataset*

In [67]:
df.loc[33,'Topics']

'angular, archiving, django, dms, document-management, document-management-system, hacktoberfest, machine-learning, ocr, optical-character-recognition, pdf'

In [68]:
# Most frequent topics

df['Topics'].dropna().str.split(",").explode().str.split().value_counts().head(15)

Topics
[machine-learning]           882
[deep-learning]              789
[python]                     728
[javascript]                 442
[android]                    411
[hacktoberfest]              401
[data-science]               397
[computer-vision]            388
[artificial-intelligence]    383
[ai]                         369
[ios]                        314
[nlp]                        313
[pytorch]                    312
[frontend]                   307
[devops]                     290
Name: count, dtype: int64

In [69]:
# Missing Descriptions

(df['Description'].str.strip() == "").sum() # Count of empty entries after removing spaces

np.int64(0)

*No missing descriptions*

In [70]:
# Checking description to determine preprocessing

df['Description'].sample(10)


4170                              该仓库主要记录 NLP 算法工程师相关的面试题
418     Tensors and Dynamic neural networks in Python ...
4428    A curated list of amazingly awesome Home Assis...
4723                       CUDA Accelerated Robot Library
2459    Blockchain explorer for Ethereum based network...
3004    🗂 The perfect Front-End Checklist for modern w...
516          AiLearning：数据分析+机器学习实战+线性代数+PyTorch+NLTK+TF2
3297                Repositorio com livros de programação
3304         Secure and easy axios integration for Nuxt 2
4468    The world's most popular open source digital s...
Name: Description, dtype: object

In [71]:
df.loc[37,'Description']

'吴恩达老师的机器学习课程个人笔记'

**PREPROCESSING**

**removing unnecessary columns**

In [72]:
cols = [
    "Repository Name",
    "Description",
    "Topics",
    "Domain",
    "Primary Language",
    "Stars Count",
    "Forks Count",
    "Updated At",
]

df = df[cols].copy()

In [73]:
df.shape

(5000, 8)

**Handling Duplicates**

In [74]:
df.drop_duplicates(inplace=True)

In [75]:
print(df.columns)

Index(['Repository Name', 'Description', 'Topics', 'Domain',
       'Primary Language', 'Stars Count', 'Forks Count', 'Updated At'],
      dtype='object')


**Checking Null**

In [76]:
df.isnull().sum()

Repository Name     0
Description         0
Topics              0
Domain              0
Primary Language    0
Stars Count         0
Forks Count         0
Updated At          0
dtype: int64

In [77]:
for col in ["Description", "Topics", "Domain", "Primary Language"]:
    print(col, (df[col].astype(str).str.strip() == "0").sum())

Description 11
Topics 180
Domain 0
Primary Language 602


In [78]:
text_cols = ["Description", "Topics", "Domain", "Primary Language"]

for col in text_cols:
    df[col] = df[col].astype(str).replace("0", "")

**Feature Engineering**

In [79]:
df['combined_text']= (
    df['Description'] + ' ' +
    df['Topics'] + ' ' +
    df['Domain'] 
)

In [80]:
df.head(1)

,Repository Name,Description,Topics,Domain,Primary Language,Stars Count,Forks Count,Updated At,combined_text
0,tensorflow,An Open Source Machine Learning Framework for ...,"deep-learning, deep-neural-networks, distribut...",Machine Learning,C++,194622,75263,2026-04-10T10:08:00Z,An Open Source Machine Learning Framework for ...


**preprocessing text**

In [81]:
import re
import string

def preprocess(text):
    text= text.lower()
    text= re.sub(r"http\S+\www\S+", "", text)   # urls
    text= text.translate(str.maketrans("","", string.punctuation))  #punctuations
    text= re.sub(r"\s+", " ", text).strip() #extra spaces

    return text

df['combined_text'] = df['combined_text'].apply(preprocess)

In [82]:
df.head(1)

,Repository Name,Description,Topics,Domain,Primary Language,Stars Count,Forks Count,Updated At,combined_text
0,tensorflow,An Open Source Machine Learning Framework for ...,"deep-learning, deep-neural-networks, distribut...",Machine Learning,C++,194622,75263,2026-04-10T10:08:00Z,an open source machine learning framework for ...


In [83]:
df.loc[0,'combined_text']

'an open source machine learning framework for everyone deeplearning deepneuralnetworks distributed machinelearning ml neuralnetwork python tensorflow machine learning'

In [84]:
df.to_csv("../data/processed/repositories.csv", index = False)